<a href="https://colab.research.google.com/github/dipanshupal516-hub/Movie-Theater-Box-Office-Concession-Analysis/blob/main/Dipanshu_pal_2025b03110132.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 15: Movie Theater Box Office & Concession Analysis
**Industry:** Entertainment / Media  
**Business scenario:** Analyze ticket sales, showtimes, genres, and snack-bar purchases to help a cinema chain optimize screen allocation and combo promotions.  
**Dataset:** `15_cinema_sales.csv`

> **Colab setup:** Upload `15_cinema_sales.csv` using the folder icon on the left (Files panel) or run the next cell.

In [1]:
# (Optional) Upload the CSV in Colab
try:
    from google.colab import files
    files.upload()   # choose 15_cinema_sales.csv
except ImportError:
    pass

Saving 15_cinema_sales.csv to 15_cinema_sales.csv


## Task 1: Load and inspect the dataset

In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.width', 200)

df = pd.read_csv('15_cinema_sales.csv')
print('Shape:', df.shape)
df.head()

Shape: (1530, 11)


,Transaction_ID,Show_Date,Movie_Title,Genre,Screen_Format,Showtime_Slot,Ticket_Quantity,Ticket_Price,Ticket_Revenue,Concession_Spend,Occupancy_Pct
0,TXN100199,2026-08-27,Midnight Whisper,Horror,Standard 2D,Evening,4,220,880,681.00,72.20
1,TXN100743,2026-10-13,Galaxy Raiders,Sci-Fi,Standard 2D,Evening,2,220,440,199.00,69.30
2,TXN100708,2026-08-29,Midnight Whisper,Horror,Standard 2D,Matinee,1,220,220,113.00,66.50
3,TXN100377,2026-09-15,Laugh Riot,Comedy,Standard 2D,Evening,3,220,660,0.00,90.90
4,TXN101029,2026-10-03,Cosmic Drift,Sci-Fi,Standard 2D,Evening,4,220,880,553.00,62.20


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1530 entries, 0 to 1529
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction_ID    1530 non-null   object 
 1   Show_Date         1530 non-null   object 
 2   Movie_Title       1530 non-null   object 
 3   Genre             1530 non-null   object 
 4   Screen_Format     1530 non-null   object 
 5   Showtime_Slot     1530 non-null   object 
 6   Ticket_Quantity   1530 non-null   int64  
 7   Ticket_Price      1530 non-null   int64  
 8   Ticket_Revenue    1530 non-null   int64  
 9   Concession_Spend  1437 non-null   float64
 10  Occupancy_Pct     1518 non-null   float64
dtypes: float64(2), int64(3), object(6)
memory usage: 131.6+ KB


In [4]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Transaction_ID,1530,1500,TXN100075,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Show_Date,1530,92,2026-09-17,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Movie_Title,1530,12,Heart Strings,150,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Genre,1530,8,Animation,265,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Screen_Format,1530,4,Standard 2D,773,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Showtime_Slot,1530,3,Evening,829,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ticket_Quantity,"1,530.00",NaN,NaN,NaN,2.89,2.33,1.00,2.00,2.00,4.00,25.00
Ticket_Price,"1,530.00",NaN,NaN,NaN,296.52,93.60,220.00,220.00,220.00,420.00,480.00
Ticket_Revenue,"1,530.00",NaN,NaN,NaN,856.59,759.40,220.00,440.00,660.00,960.00,"11,040.00"
Concession_Spend,"1,437.00",NaN,NaN,NaN,295.90,356.53,0.00,0.00,246.00,421.00,"5,175.00"


## Task 2: Identify missing values and duplicate `Transaction_ID` records

In [5]:
missing = pd.DataFrame({'Missing_Count': df.isna().sum(),
                        'Missing_%': (df.isna().mean() * 100).round(2)})
print(missing)

dup_mask = df.duplicated(subset='Transaction_ID', keep=False)
print('\nRows involved in duplicate Transaction_IDs:', dup_mask.sum())
print('Extra (redundant) duplicate rows:', df.duplicated(subset='Transaction_ID').sum())
df[dup_mask].sort_values('Transaction_ID').head(10)

                  Missing_Count  Missing_%
Transaction_ID                0       0.00
Show_Date                     0       0.00
Movie_Title                   0       0.00
Genre                         0       0.00
Screen_Format                 0       0.00
Showtime_Slot                 0       0.00
Ticket_Quantity               0       0.00
Ticket_Price                  0       0.00
Ticket_Revenue                0       0.00
Concession_Spend             93       6.08
Occupancy_Pct                12       0.78

Rows involved in duplicate Transaction_IDs: 60
Extra (redundant) duplicate rows: 30


,Transaction_ID,Show_Date,Movie_Title,Genre,Screen_Format,Showtime_Slot,Ticket_Quantity,Ticket_Price,Ticket_Revenue,Concession_Spend,Occupancy_Pct
20,TXN100075,2026-08-12,Galaxy Raiders,Sci-Fi,Standard 2D,Matinee,2,220,440,0.00,53.60
643,TXN100075,2026-08-12,Galaxy Raiders,Sci-Fi,Standard 2D,Matinee,2,220,440,0.00,53.60
702,TXN100091,2026-09-30,Cosmic Drift,Sci-Fi,3D,Matinee,2,300,600,223.00,53.70
502,TXN100091,2026-09-30,Cosmic Drift,Sci-Fi,3D,Matinee,2,300,600,223.00,53.70
737,TXN100101,2026-08-02,Haunted Manor,Horror,3D,Matinee,5,300,1500,0.00,80.50
656,TXN100101,2026-08-02,Haunted Manor,Horror,3D,Matinee,5,300,1500,0.00,80.50
827,TXN100102,2026-08-26,Jungle Quest,Animation,Standard 2D,Evening,6,220,1320,NaN,64.80
816,TXN100102,2026-08-26,Jungle Quest,Animation,Standard 2D,Evening,6,220,1320,NaN,64.80
53,TXN100169,2026-08-27,Little Stars,Animation,4DX,Late Night,2,480,960,311.00,71.20
1267,TXN100169,2026-08-27,Little Stars,Animation,4DX,Late Night,2,480,960,311.00,71.20


In [6]:
# Remove duplicates (keep the first occurrence of each Transaction_ID)
df = df.drop_duplicates(subset='Transaction_ID', keep='first').reset_index(drop=True)
print('Shape after removing duplicates:', df.shape)

Shape after removing duplicates: (1500, 11)


## Task 3: Clean missing `Concession_Spend` values
**Strategy:** Concession spend is right-skewed and many customers spend 0, so the *mean* would be distorted by large orders. I fill missing values with the **median Concession_Spend of the same Genre** (genres show different snack behaviour, e.g. Animation vs Drama). Any leftover gaps fall back to the overall median. Missing `Occupancy_Pct` is filled with the median of its Screen_Format for the same reason.

In [7]:
print('Missing Concession_Spend before:', df['Concession_Spend'].isna().sum())

df['Concession_Spend'] = df['Concession_Spend'].fillna(
    df.groupby('Genre')['Concession_Spend'].transform('median'))
df['Concession_Spend'] = df['Concession_Spend'].fillna(df['Concession_Spend'].median())

# Occupancy_Pct - fill by screen-format median
df['Occupancy_Pct'] = df['Occupancy_Pct'].fillna(
    df.groupby('Screen_Format')['Occupancy_Pct'].transform('median'))

print('Missing Concession_Spend after :', df['Concession_Spend'].isna().sum())
print('Missing Occupancy_Pct after    :', df['Occupancy_Pct'].isna().sum())

Missing Concession_Spend before: 90
Missing Concession_Spend after : 0
Missing Occupancy_Pct after    : 0


## Task 4: Convert `Show_Date` to datetime and extract weekday vs weekend indicators

In [8]:
df['Show_Date'] = pd.to_datetime(df['Show_Date'])
df['Day_Name'] = df['Show_Date'].dt.day_name()
df['Weekday_Num'] = df['Show_Date'].dt.dayofweek          # Mon=0 ... Sun=6

# Hint: np.where() to flag weekend days (Saturday, Sunday)
df['Is_Weekend'] = np.where(df['Day_Name'].isin(['Saturday', 'Sunday']), 1, 0)
df['Day_Type'] = np.where(df['Is_Weekend'] == 1, 'Weekend', 'Weekday')

df[['Show_Date', 'Day_Name', 'Is_Weekend', 'Day_Type']].head()

,Show_Date,Day_Name,Is_Weekend,Day_Type
0,2026-08-27,Thursday,0,Weekday
1,2026-10-13,Tuesday,0,Weekday
2,2026-08-29,Saturday,1,Weekend
3,2026-09-15,Tuesday,0,Weekday
4,2026-10-03,Saturday,1,Weekend


## Task 5: Total ticket revenue, total concession revenue, and combined revenue

In [9]:
df['Total_Spend'] = df['Ticket_Revenue'] + df['Concession_Spend']

total_ticket = df['Ticket_Revenue'].sum()
total_conc   = df['Concession_Spend'].sum()
combined     = total_ticket + total_conc

revenue_summary = pd.DataFrame({
    'Revenue_Type': ['Ticket Revenue', 'Concession Revenue', 'Combined Revenue'],
    'Amount': [total_ticket, total_conc, combined],
    'Share_%': [total_ticket/combined*100, total_conc/combined*100, 100]})
revenue_summary

,Revenue_Type,Amount,Share_%
0,Ticket Revenue,"1,283,720.00",74.59
1,Concession Revenue,"437,427.00",25.41
2,Combined Revenue,"1,721,147.00",100.00


## Task 6: Top 10 movies by total ticket sales and ticket revenue

In [10]:
movie_perf = (df.groupby('Movie_Title')
                .agg(Total_Tickets_Sold=('Ticket_Quantity', 'sum'),
                     Total_Ticket_Revenue=('Ticket_Revenue', 'sum'),
                     Total_Concession=('Concession_Spend', 'sum'),
                     Transactions=('Transaction_ID', 'count')))

print('Top 10 by tickets sold')
top_by_tickets = movie_perf.sort_values('Total_Tickets_Sold', ascending=False).head(10)
display(top_by_tickets)

print('Top 10 by ticket revenue')
top_by_revenue = movie_perf.sort_values('Total_Ticket_Revenue', ascending=False).head(10)
display(top_by_revenue)

Top 10 by tickets sold


,Total_Tickets_Sold,Total_Ticket_Revenue,Total_Concession,Transactions
Movie_Title,,,,
Heart Strings,427,126960,"36,789.00",148
Galaxy Raiders,394,111980,"46,744.50",139
The Last Heist,384,117020,"35,552.00",126
Neon Sprint,380,111780,"43,448.00",123
Haunted Manor,377,109220,"33,002.00",122
Jungle Quest,364,107340,"53,168.00",128
Midnight Whisper,364,103960,"27,603.00",127
Little Stars,362,107820,"41,594.00",128
Dust & Glory,340,103220,"24,576.00",118


Top 10 by ticket revenue


,Total_Tickets_Sold,Total_Ticket_Revenue,Total_Concession,Transactions
Movie_Title,,,,
Heart Strings,427,126960,"36,789.00",148
The Last Heist,384,117020,"35,552.00",126
Galaxy Raiders,394,111980,"46,744.50",139
Neon Sprint,380,111780,"43,448.00",123
Haunted Manor,377,109220,"33,002.00",122
Little Stars,362,107820,"41,594.00",128
Jungle Quest,364,107340,"53,168.00",128
Laugh Riot,333,105400,"30,781.00",127
Midnight Whisper,364,103960,"27,603.00",127


## Task 7: Average occupancy % across screen formats

In [11]:
# Hint: groupby('Screen_Format').mean(numeric_only=True)
format_summary = (df.groupby('Screen_Format')
                    .agg(Avg_Occupancy_Pct=('Occupancy_Pct', 'mean'),
                         Avg_Ticket_Price=('Ticket_Price', 'mean'),
                         Total_Ticket_Revenue=('Ticket_Revenue', 'sum'),
                         Transactions=('Transaction_ID', 'count'))
                    .sort_values('Avg_Occupancy_Pct', ascending=False))
format_summary

,Avg_Occupancy_Pct,Avg_Ticket_Price,Total_Ticket_Revenue,Transactions
Screen_Format,,,,
IMAX,82.24,420.00,288960,234
4DX,74.86,480.00,209760,154
3D,67.41,300.00,305400,354
Standard 2D,60.56,220.00,479600,758


In [12]:
# Same result using the hint exactly as written
df.groupby('Screen_Format').mean(numeric_only=True)[['Occupancy_Pct']].sort_values('Occupancy_Pct', ascending=False)

,Occupancy_Pct
Screen_Format,
IMAX,82.24
4DX,74.86
3D,67.41
Standard 2D,60.56


## Task 8: Ticket sales and concession spending - weekdays vs weekends

In [13]:
daytype_summary = (df.groupby('Day_Type')
                     .agg(Transactions=('Transaction_ID', 'count'),
                          Tickets_Sold=('Ticket_Quantity', 'sum'),
                          Total_Ticket_Revenue=('Ticket_Revenue', 'sum'),
                          Total_Concession=('Concession_Spend', 'sum'),
                          Avg_Ticket_Revenue=('Ticket_Revenue', 'mean'),
                          Avg_Concession=('Concession_Spend', 'mean'),
                          Avg_Occupancy_Pct=('Occupancy_Pct', 'mean')))
daytype_summary

,Transactions,Tickets_Sold,Total_Ticket_Revenue,Total_Concession,Avg_Ticket_Revenue,Avg_Concession,Avg_Occupancy_Pct
Day_Type,,,,,,,
Weekday,1055,3060,905560,"286,301.50",858.35,271.38,64.55
Weekend,445,1263,378160,"151,125.50",849.80,339.61,72.90


In [14]:
# Per-day averages (fair comparison since there are 5 weekdays vs 2 weekend days)
days_count = df.groupby('Day_Type')['Show_Date'].nunique()
daytype_summary['Distinct_Days'] = days_count
daytype_summary['Ticket_Revenue_per_Day'] = daytype_summary['Total_Ticket_Revenue'] / daytype_summary['Distinct_Days']
daytype_summary['Concession_per_Day'] = daytype_summary['Total_Concession'] / daytype_summary['Distinct_Days']
daytype_summary[['Distinct_Days', 'Ticket_Revenue_per_Day', 'Concession_per_Day']]

,Distinct_Days,Ticket_Revenue_per_Day,Concession_per_Day
Day_Type,,,
Weekday,65,"13,931.69","4,404.64"
Weekend,27,"14,005.93","5,597.24"


## Task 9: Attendance across showtime slots (Matinee, Evening, Late Night)

In [15]:
slot_order = ['Matinee', 'Evening', 'Late Night']
slot_summary = (df.groupby('Showtime_Slot')
                  .agg(Transactions=('Transaction_ID', 'count'),
                       Tickets_Sold=('Ticket_Quantity', 'sum'),
                       Avg_Occupancy_Pct=('Occupancy_Pct', 'mean'),
                       Total_Ticket_Revenue=('Ticket_Revenue', 'sum'),
                       Avg_Concession=('Concession_Spend', 'mean'))
                  .reindex(slot_order))
slot_summary['Ticket_Share_%'] = slot_summary['Tickets_Sold'] / slot_summary['Tickets_Sold'].sum() * 100
slot_summary

,Transactions,Tickets_Sold,Avg_Occupancy_Pct,Total_Ticket_Revenue,Avg_Concession,Ticket_Share_%
Showtime_Slot,,,,,,
Matinee,384,1115,60.84,324900,279.36,25.79
Evening,814,2387,71.00,719540,291.36,55.22
Late Night,302,821,64.18,239280,307.91,18.99


## Task 10: Concession attach rate (% of transactions with concession spend > 0)

In [16]:
df['Has_Concession'] = np.where(df['Concession_Spend'] > 0, 1, 0)
overall_attach = df['Has_Concession'].mean() * 100
print(f'Overall concession attach rate: {overall_attach:.2f}%')

attach_by_daytype = (df.groupby('Day_Type')['Has_Concession'].mean() * 100).round(2)
attach_by_slot = (df.groupby('Showtime_Slot')['Has_Concession'].mean() * 100).reindex(slot_order).round(2)
print('\nBy day type:\n', attach_by_daytype)
print('\nBy showtime slot:\n', attach_by_slot)

Overall concession attach rate: 72.00%

By day type:
 Day_Type
Weekday   69.19
Weekend   78.65
Name: Has_Concession, dtype: float64

By showtime slot:
 Showtime_Slot
Matinee      70.05
Evening      71.87
Late Night   74.83
Name: Has_Concession, dtype: float64


## Task 11: Average ticket and snack spending by movie genre

In [17]:
genre_summary = (df.groupby('Genre')
                   .agg(Avg_Ticket_Spend=('Ticket_Revenue', 'mean'),
                        Avg_Snack_Spend=('Concession_Spend', 'mean'),
                        Avg_Total_Spend=('Total_Spend', 'mean'),
                        Attach_Rate_pct=('Has_Concession', lambda s: s.mean()*100),
                        Transactions=('Transaction_ID', 'count'))
                   .sort_values('Avg_Snack_Spend', ascending=False))
genre_summary

,Avg_Ticket_Spend,Avg_Snack_Spend,Avg_Total_Spend,Attach_Rate_pct,Transactions
Genre,,,,,
Animation,840.47,370.16,"1,210.63",72.66,256
Action,868.46,327.13,"1,195.58",77.09,227
Sci-Fi,825.86,321.70,"1,147.57",73.49,249
Thriller,928.73,282.16,"1,210.89",72.22,126
Romance,857.84,248.57,"1,106.41",73.65,148
Horror,856.14,243.39,"1,099.54",65.06,249
Comedy,829.92,242.37,"1,072.29",70.87,127
Drama,874.75,208.27,"1,083.02",71.19,118


## Task 12: NumPy - mean, median, and standard deviation of total transaction spend

In [18]:
spend = df['Total_Spend'].to_numpy()
print(f'Mean   : {np.mean(spend):,.2f}')
print(f'Median : {np.median(spend):,.2f}')
print(f'Std Dev: {np.std(spend, ddof=1):,.2f}  (sample std)')
print(f'90th percentile: {np.percentile(spend, 90):,.2f}')

Mean   : 1,147.43
Median : 900.00
Std Dev: 974.46  (sample std)
90th percentile: 2,000.10


## Task 13: Unusually large group bookings using the IQR method on ticket quantity

In [19]:
q1 = df['Ticket_Quantity'].quantile(0.25)
q3 = df['Ticket_Quantity'].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
print(f'Q1={q1}, Q3={q3}, IQR={iqr}, Lower bound={lower}, Upper bound={upper}')

large_groups = df[df['Ticket_Quantity'] > upper]
print('Number of outlier (large group) bookings:', len(large_groups))
large_groups[['Transaction_ID', 'Show_Date', 'Movie_Title', 'Screen_Format',
              'Ticket_Quantity', 'Ticket_Revenue', 'Concession_Spend']].sort_values('Ticket_Quantity', ascending=False).head(10)

Q1=2.0, Q3=4.0, IQR=2.0, Lower bound=-1.0, Upper bound=7.0
Number of outlier (large group) bookings: 25


,Transaction_ID,Show_Date,Movie_Title,Screen_Format,Ticket_Quantity,Ticket_Revenue,Concession_Spend
180,TXN100107,2026-08-17,Little Stars,Standard 2D,25,5500,309.00
202,TXN100749,2026-10-31,The Last Heist,IMAX,24,10080,"2,820.00"
1026,TXN101137,2026-10-18,Haunted Manor,Standard 2D,24,5280,"3,798.00"
759,TXN101463,2026-09-28,Haunted Manor,Standard 2D,24,5280,"3,764.00"
813,TXN101454,2026-10-22,Dust & Glory,Standard 2D,24,5280,0.00
429,TXN100175,2026-08-04,Jungle Quest,Standard 2D,23,5060,"5,175.00"
543,TXN100142,2026-09-08,The Last Heist,4DX,23,11040,0.00
68,TXN100566,2026-08-20,Laugh Riot,IMAX,22,9240,0.00
161,TXN100975,2026-10-19,Neon Sprint,3D,21,6300,0.00
1390,TXN101281,2026-08-19,Heart Strings,Standard 2D,19,4180,"2,551.00"


In [20]:
print('Outlier bookings - share of tickets  : {:.2f}%'.format(large_groups['Ticket_Quantity'].sum()/df['Ticket_Quantity'].sum()*100))
print('Outlier bookings - share of ticket rev: {:.2f}%'.format(large_groups['Ticket_Revenue'].sum()/df['Ticket_Revenue'].sum()*100))
print('Most common formats for large groups:\n', large_groups['Screen_Format'].value_counts())

Outlier bookings - share of tickets  : 9.88%
Outlier bookings - share of ticket rev: 9.53%
Most common formats for large groups:
 Screen_Format
Standard 2D    14
3D              5
IMAX            4
4DX             2
Name: count, dtype: int64


## Task 14: Ticket-quantity bands (Solo, Couple, Small Group, Large Group)

In [21]:
# Hint: pd.cut()
bins = [0, 1, 2, 5, np.inf]
labels = ['Solo', 'Couple', 'Small Group', 'Large Group']
df['Group_Band'] = pd.cut(df['Ticket_Quantity'], bins=bins, labels=labels)

band_summary = (df.groupby('Group_Band', observed=True)
                  .agg(Transactions=('Transaction_ID', 'count'),
                       Tickets_Sold=('Ticket_Quantity', 'sum'),
                       Avg_Ticket_Revenue=('Ticket_Revenue', 'mean'),
                       Avg_Concession=('Concession_Spend', 'mean'),
                       Attach_Rate_pct=('Has_Concession', lambda s: s.mean()*100)))
band_summary['Transaction_Share_%'] = band_summary['Transactions']/band_summary['Transactions'].sum()*100
band_summary

,Transactions,Tickets_Sold,Avg_Ticket_Revenue,Avg_Concession,Attach_Rate_pct,Transaction_Share_%
Group_Band,,,,,,
Solo,253,253,290.12,115.29,73.91,16.87
Couple,582,1164,598.69,205.60,71.48,38.80
Small Group,592,2191,"1,103.58",376.17,71.62,39.47
Large Group,73,715,"2,856.99",902.80,72.60,4.87


## Extra analysis: correlation and best/worst combinations

In [22]:
print('Correlation (Ticket_Quantity vs Concession_Spend):',
      round(df['Ticket_Quantity'].corr(df['Concession_Spend']), 3))
print('Correlation (Occupancy_Pct vs Ticket_Revenue):',
      round(df['Occupancy_Pct'].corr(df['Ticket_Revenue']), 3))

combo = (df.groupby(['Screen_Format', 'Showtime_Slot'])['Occupancy_Pct']
           .mean().unstack().reindex(columns=slot_order))
combo

Correlation (Ticket_Quantity vs Concession_Spend): 0.573
Correlation (Occupancy_Pct vs Ticket_Revenue): 0.194


Showtime_Slot,Matinee,Evening,Late Night
Screen_Format,,,
3D,60.25,70.66,67.19
4DX,68.20,79.13,72.89
IMAX,75.35,86.71,78.86
Standard 2D,54.80,64.52,57.62


In [23]:
# Weekend vs weekday occupancy by showtime slot
df.pivot_table(index='Showtime_Slot', columns='Day_Type', values='Occupancy_Pct', aggfunc='mean').reindex(slot_order)

Day_Type,Weekday,Weekend
Showtime_Slot,,
Matinee,58.02,66.88
Evening,68.65,76.69
Late Night,61.53,70.95


## Task 15: Marketing and scheduling recommendations
*The cell below prints the key figures that support each recommendation.*

In [24]:
best_format   = format_summary['Avg_Occupancy_Pct'].idxmax()
best_slot    = slot_summary['Avg_Occupancy_Pct'].idxmax()
weak_slot    = slot_summary['Avg_Occupancy_Pct'].idxmin()
top_snack_g  = genre_summary['Avg_Snack_Spend'].idxmax()
low_snack_g  = genre_summary['Avg_Snack_Spend'].idxmin()
top_movie    = top_by_revenue.index[0]

print('Best-filled format        :', best_format, f"({format_summary.loc[best_format,'Avg_Occupancy_Pct']:.1f}% occupancy)")
print('Best / weakest slot       :', best_slot, '/', weak_slot)
print('Highest / lowest snack genre:', top_snack_g, '/', low_snack_g)
print('Top movie by ticket revenue :', top_movie)
print('Weekend vs weekday concession attach rate (%):', attach_by_daytype.to_dict())
print('Large-group share of tickets (%):', round(large_groups['Ticket_Quantity'].sum()/df['Ticket_Quantity'].sum()*100, 2))

Best-filled format        : IMAX (82.2% occupancy)
Best / weakest slot       : Evening / Matinee
Highest / lowest snack genre: Animation / Drama
Top movie by ticket revenue : Heart Strings
Weekend vs weekday concession attach rate (%): {'Weekday': 69.19, 'Weekend': 78.65}
Large-group share of tickets (%): 9.88


### Recommendations
1. **Screen allocation:** Give premium formats (IMAX / 4DX) the highest-demand titles and extra evening/weekend shows, since they achieve the best occupancy and the highest revenue per ticket, while reducing low-occupancy Standard 2D screens for weaker titles.
2. **Showtime scheduling:** Evening slots draw the largest audience, so place new releases and top-ranked movies there; use Matinee and Late Night slots for discounted pricing, family titles (matinee) and horror/thriller titles (late night) to lift occupancy.
3. **Weekend strategy:** Weekends show higher occupancy and a higher concession attach rate, so schedule extra screenings and run premium combo deals on Saturdays and Sundays; use weekday-only offers (e.g., "Tuesday value combo") to lift weekday traffic.
4. **Genre-targeted combos:** Animation and Action audiences spend the most on snacks - promote family/kids' combos and large sharing bundles for these genres, and offer lower-priced add-on deals for Drama and Romance audiences, who spend the least on concessions.
5. **Group promotions:** Couples and Small Groups make up most transactions, while Large Group bookings are rare but high-value outliers; launch "couple combos" and "group bundles" (ticket + shared popcorn/drinks), and create a pre-booking desk/corporate offer for large groups.

## Summary of deliverables
- **Top movies table** - Task 6
- **Weekend vs weekday comparison** - Task 8
- **Screen format summary** - Task 7
- **Concession attach metrics** - Tasks 10, 11
- **Recommendations** - Task 15